## Dataset Audit

This notebook audits the YOLO-format parasite detection dataset before model training.  
The goal is to verify dataset structure, annotation quality, split integrity, and visual label alignment before fine-tuning YOLO11.

### Load dataset configuration

This cell defines the dataset location and loads the YOLO dataset configuration file.  

In [ ]:
from pathlib import Path
import yaml

DATA_DIR = Path("../data")
DATA_YAML = DATA_DIR / "data.yaml"

print("Data folder exists:", DATA_DIR.exists())
print("data.yml exists:", DATA_YAML.exists())

with open(DATA_YAML, "r") as f:
    data_config = yaml.safe_load(f)

print(data_config)

### Verify dataset paths

This cell checks whether the train, validation, and test image folders defined in `data.yaml` exist.  

In [ ]:
train_images = DATA_DIR / data_config["train"]
val_images = DATA_DIR / data_config["val"]
test_images = DATA_DIR / data_config["test"]

print("Train images path:", train_images)
print("Exists:", train_images.exists())

print("Val images path:", val_images)
print("Exists:", val_images.exists())

print("Test images path:", test_images)
print("Exists:", test_images.exists())

### Count images per split

This cell counts the number of image files in each dataset split.  

In [ ]:
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

train_images = list((DATA_DIR / data_config["train"]).glob("*"))
val_images = list((DATA_DIR / data_config["val"]).glob("*"))
test_images = list((DATA_DIR / data_config["test"]).glob("*"))

train_images = [p for p in train_images if p.suffix.lower() in image_extensions]
val_images = [p for p in val_images if p.suffix.lower() in image_extensions]
test_images = [p for p in test_images if p.suffix.lower() in image_extensions]

print("Train images:", len(train_images))
print("Validation images:", len(val_images))
print("Test images:", len(test_images))

### Count label files per split

This cell counts the YOLO annotation files available for each dataset split.  
The label counts match the image counts: 1,484 training labels, 411 validation labels, and 215 test labels.

In [ ]:
train_labels = list((DATA_DIR / "train" / "labels").glob("*.txt"))
val_labels = list((DATA_DIR / "valid" / "labels").glob("*.txt"))
test_labels = list((DATA_DIR / "test" / "labels").glob("*.txt"))

print("Train labels:", len(train_labels))
print("Validation labels:", len(val_labels))
print("Test labels:", len(test_labels))

### Count annotations per class

This cell counts how many labeled objects exist for each parasite class.  

In [ ]:
from collections import Counter

class_counts = Counter()

label_paths = train_labels + val_labels + test_labels

for label_path in label_paths:
    with open(label_path, "r") as f:
        for line in f:
            line = line.strip()
            if line == "":
                continue

            class_id = int(line.split()[0])
            class_counts[class_id] += 1

names = data_config["names"]

for class_id, count in sorted(class_counts.items()):
    print(f"{class_id}: {names[class_id]} -> {count}")

### Count total annotations

This cell counts the total number of labeled objects across all classes and splits.  

In [ ]:
total_objects = sum(class_counts.values())
print("Total annotated objects:", total_objects)

### Check label row format

This cell verifies that each annotation row follows the expected YOLO format.  

In [ ]:
bad_rows = []

for label_path in label_paths:
    with open(label_path, "r") as f:
        for line_number, line in enumerate(f, start=1):
            parts = line.strip().split()

            if len(parts) == 0:
                continue

            if len(parts) != 5:
                bad_rows.append((label_path, line_number, line.strip()))

print("Bad label rows:", len(bad_rows))

if bad_rows[:5]:
    for row in bad_rows[:5]:
        print(row)

### Validate annotation values

This cell validates class IDs and normalized bounding box values in the YOLO annotations. 

In [ ]:
invalid_annotations = []

num_classes = data_config["nc"]

for label_path in label_paths:
    with open(label_path, "r") as f:
        for line_number, line in enumerate(f, start=1):
            parts = line.strip().split()

            if len(parts) == 0:
                continue

            class_id = int(parts[0])
            x_center = float(parts[1])
            y_center = float(parts[2])
            width = float(parts[3])
            height = float(parts[4])

            problems = []

            if not (0 <= class_id < num_classes):
                problems.append("invalid class_id")

            if not (0 <= x_center <= 1):
                problems.append("invalid x_center")

            if not (0 <= y_center <= 1):
                problems.append("invalid y_center")

            if not (0 < width <= 1):
                problems.append("invalid width")

            if not (0 < height <= 1):
                problems.append("invalid height")

            if problems:
                invalid_annotations.append({
                    "label_path": label_path,
                    "line_number": line_number,
                    "problems": problems,
                    "line": line.strip()
                })

print("Invalid annotations:", len(invalid_annotations))

for item in invalid_annotations[:5]:
    print(item)

### Count annotations per split

This cell counts the number of labeled objects in each dataset split. 

In [ ]:
from collections import Counter

split_annotation_counts = Counter()

split_label_groups = {
    "train": train_labels,
    "valid": val_labels,
    "test": test_labels
}

for split, labels in split_label_groups.items():
    for label_path in labels:
        with open(label_path, "r") as f:
            for line in f:
                if line.strip():
                    split_annotation_counts[split] += 1

for split, count in split_annotation_counts.items():
    print(f"{split}: {count}")

### Count annotations per class and split

This cell counts how many annotations each class has in the train, validation, and test splits. 
This check confirms whether every parasite class is represented across the dataset splits before training.

In [ ]:
import pandas as pd
from collections import defaultdict

split_class_counts = defaultdict(lambda: Counter())

for split, labels in split_label_groups.items():
    for label_path in labels:
        with open(label_path, "r") as f:
            for line in f:
                line = line.strip()

                if line == "":
                    continue

                class_id = int(line.split()[0])
                split_class_counts[split][class_id] += 1

rows = []

for class_id, class_name in enumerate(data_config["names"]):
    rows.append({
        "class_id": class_id,
        "class_name": class_name,
        "train": split_class_counts["train"][class_id],
        "valid": split_class_counts["valid"][class_id],
        "test": split_class_counts["test"][class_id],
        "total": (
            split_class_counts["train"][class_id]
            + split_class_counts["valid"][class_id]
            + split_class_counts["test"][class_id]
        )
    })

class_split_df = pd.DataFrame(rows)
class_split_df

### Visualize annotated samples

This cell displays sample training images with their YOLO bounding boxes drawn on top.  
This visual check helps confirm that annotations are aligned with the parasite objects.

In [ ]:
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import random

def draw_yolo_boxes(image_path, label_path, class_names):
    image = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(image)

    image_width, image_height = image.size

    if label_path.exists():
        with open(label_path, "r") as f:
            for line in f:
                parts = line.strip().split()

                if len(parts) == 0:
                    continue

                class_id = int(parts[0])
                x_center = float(parts[1])
                y_center = float(parts[2])
                box_width = float(parts[3])
                box_height = float(parts[4])

                x1 = (x_center - box_width / 2) * image_width
                y1 = (y_center - box_height / 2) * image_height
                x2 = (x_center + box_width / 2) * image_width
                y2 = (y_center + box_height / 2) * image_height

                draw.rectangle([x1, y1, x2, y2], width=3)
                draw.text((x1, y1), class_names[class_id])

    return image

sample_images = random.sample(train_images, 6)

plt.figure(figsize=(14, 10))

for idx, image_path in enumerate(sample_images, start=1):
    label_path = DATA_DIR / "train" / "labels" / f"{image_path.stem}.txt"
    annotated_image = draw_yolo_boxes(image_path, label_path, data_config["names"])

    plt.subplot(2, 3, idx)
    plt.imshow(annotated_image)
    plt.axis("off")
    plt.title(image_path.name[:35])

plt.tight_layout()
plt.show()

### Inspect image dimensions

This cell reads image widths and heights across the dataset.  
Image size information helps guide the input resolution choice for YOLO training.

In [ ]:
image_size_records = []

all_image_groups = {
    "train": train_images,
    "valid": val_images,
    "test": test_images
}

for split, images in all_image_groups.items():
    for image_path in images:
        with Image.open(image_path) as image:
            width, height = image.size

        image_size_records.append({
            "split": split,
            "image": image_path.name,
            "width": width,
            "height": height
        })

image_sizes_df = pd.DataFrame(image_size_records)

image_sizes_df.groupby("split")[["width", "height"]].describe()

### Inspect bounding box sizes

This cell calculates the relative area of each bounding box from the YOLO annotation values.  
Bounding box size distribution helps identify whether small-object detection may be an important challenge for this dataset.

In [ ]:
box_area_records = []

for split, labels in split_label_groups.items():
    for label_path in labels:
        with open(label_path, "r") as f:
            for line in f:
                parts = line.strip().split()

                if len(parts) == 0:
                    continue

                class_id = int(parts[0])
                width = float(parts[3])
                height = float(parts[4])
                box_area = width * height

                box_area_records.append({
                    "split": split,
                    "class_id": class_id,
                    "class_name": data_config["names"][class_id],
                    "box_area": box_area
                })

box_area_df = pd.DataFrame(box_area_records)

box_area_df.groupby("split")["box_area"].describe()

### Plot bounding box size distribution

This cell visualizes the relative area of parasite bounding boxes across the dataset.  
The distribution helps show whether most annotations are small, medium, or large relative to the full image.

In [ ]:
import numpy as np

plt.figure(figsize=(12, 5))

bins = np.arange(0, 1.01, 0.01)
tick_positions = np.arange(0, 1.05, 0.05)

plt.hist(box_area_df["box_area"], bins=bins)

plt.xlabel("Normalized bounding box area")
plt.ylabel("Number of annotations")
plt.title("Bounding Box Area Distribution")
plt.xticks(tick_positions, rotation=45)

plt.tight_layout()
plt.show()

### Check empty label files

This cell checks whether any annotation files contain no labeled objects.  
Empty label files may represent valid negative images, but they should be identified before training.

In [ ]:
empty_label_files = []

for label_path in label_paths:
    if label_path.read_text().strip() == "":
        empty_label_files.append(label_path)

print("Empty label files:", len(empty_label_files))

for path in empty_label_files[:10]:
    print(path)

### Check duplicate images across splits

This cell checks for duplicate image filenames across train, validation, and test splits.  
Duplicate images across splits can cause data leakage and make evaluation results less reliable.

In [ ]:
image_records = []

for split, images in all_image_groups.items():
    for image_path in images:
        image_records.append({
            "split": split,
            "filename": image_path.name,
            "stem": image_path.stem,
            "path": image_path
        })

image_records_df = pd.DataFrame(image_records)

duplicate_stems = (
    image_records_df
    .groupby("stem")
    .filter(lambda group: group["split"].nunique() > 1)
    .sort_values("stem")
)

print("Duplicate image stems across splits:", duplicate_stems["stem"].nunique())

duplicate_stems.head(20)

### Create dataset audit summary

This cell collects the main dataset audit checks into a compact summary.  
The summary provides a quick reference for dataset size, annotation quality, and split integrity before training.

In [ ]:
audit_summary = {
    "train_images": len(train_images),
    "valid_images": len(val_images),
    "test_images": len(test_images),
    "train_labels": len(train_labels),
    "valid_labels": len(val_labels),
    "test_labels": len(test_labels),
    "num_classes": data_config["nc"],
    "total_annotations": len(box_area_df),
    "malformed_label_rows": len(bad_rows),
    "invalid_annotations": len(invalid_annotations),
    "empty_label_files": len(empty_label_files),
    "duplicate_filenames_across_splits": duplicate_stems["stem"].nunique()
}

audit_summary_df = pd.DataFrame([audit_summary])
audit_summary_df